# Pré-computação VarNet brain 4× — passo 5 do S4

Roda inferência da E2E-VarNet pré-treinada (Sriram et al., 2020) sobre os 352 volumes elegíveis (213 train + 46 val + 46 cal + 47 test), salvando `{recon, target, error_map, max_val}` por volume em `data/recons/{split}/{stem}.npz`.

**Tempo estimado:** ~1h em T4 (single session, dentro do limite de 9h do Kaggle Free).

**Pré-requisitos:**
1. Conta Kaggle Free com acesso a GPU T4.
2. 4 Datasets privados já uploaded (slugs começando com `fastmri-brain-`):
   - `fastmri-brain-axflair-part-a`
   - `fastmri-brain-axflair-part-b`
   - `fastmri-brain-axt1`
   - `fastmri-brain-axt1post`
3. Repositório `KR0N0S7/tcc-mri-uncertainty` público no GitHub.
4. Os 4 datasets adicionados como Input via **Settings → Add Input → Datasets**.

**Em caso de timeout/interrupção:** apenas reabra o notebook e rode tudo de novo. O script de pré-computação é resumível — pula volumes que já têm `.npz` salvo.

## 1. Setup do ambiente

In [ ]:
# Verifica que estamos numa GPU T4
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Clona o repositório (precisa estar publico)
!git clone https://github.com/KR0N0S7/tcc-mri-uncertainty.git
%cd tcc-mri-uncertainty

In [ ]:
# Instala dependencias (algumas ja vem no ambiente Kaggle)
!pip install -q fastmri==0.3.0 python-dotenv tqdm h5py

## 2. Unifica os 4 datasets num único diretório

Os datasets foram montados em `/kaggle/input/fastmri-brain-*/`. O `SliceDataset` do `fastmri` espera um `root` único — criamos symlinks (zero cópia, zero disco extra) num diretório de trabalho.

A cell abaixo **auto-descobre** todos os datasets que começam com `fastmri-brain-`. Se aparecer `Esperado 352, encontrou X`, verifica em **Settings → Datasets** se os 4 estão adicionados.

In [ ]:
from pathlib import Path
import os

TARGET = Path('/kaggle/working/anotados')
TARGET.mkdir(parents=True, exist_ok=True)

# Auto-descobre datasets brain montados pelo Kaggle.
kaggle_input = Path('/kaggle/input')
SOURCES = sorted(d for d in kaggle_input.glob('fastmri-brain-*') if d.is_dir())

print(f'Datasets encontrados: {len(SOURCES)}')
for d in SOURCES:
    n_h5 = len(list(d.glob('*.h5')))
    print(f'  {d.name}: {n_h5} arquivos .h5')

if not SOURCES:
    raise RuntimeError(
        'Nenhum dataset brain montado em /kaggle/input/. '
        'Adicione via Settings -> Add Input -> Datasets.'
    )

# Cria symlinks para todos os .h5 dos datasets descobertos
for src in SOURCES:
    for h5 in src.glob('*.h5'):
        dst = TARGET / h5.name
        if not dst.exists():
            os.symlink(h5, dst)

n_files = len(list(TARGET.glob('*.h5')))
print(f'\n{n_files} symlinks em {TARGET}')
assert n_files == 352, (
    f'Esperado 352 .h5, encontrado {n_files}. '
    f'Verifica se TODOS os 4 datasets foram adicionados ao notebook.'
)

## 3. Configura variáveis de ambiente do projeto

In [ ]:
import os

os.environ['TCC_ANOTADOS_DIR'] = '/kaggle/working/anotados'
os.environ['TCC_BRAIN_CSV']    = '/dev/null'  # nao usado neste passo
os.environ['TCC_RECONS_DIR']   = '/kaggle/working/recons'
os.environ['TCC_SPLITS_DIR']   = '/kaggle/working/tcc-mri-uncertainty/splits'

!python -m src.config

## 4. Baixa o checkpoint VarNet brain 4×

In [ ]:
!python scripts/download_checkpoint.py

In [ ]:
# Verifica que carrega e bate com o SHA-256 do MANIFEST
!python -c "from src.models import load_pretrained_varnet; m, s = load_pretrained_varnet('checkpoints/brain_leaderboard_state_dict.pt'); n = sum(p.numel() for p in m.parameters()); print(f'SHA-256: {s}'); print(f'Parametros: {n:,}'); print(f'Esperado: 2cdfdde2fe662f8995316a7ba981d8d3f405d2a5fd82a61bfa1bc4e56062262f / 29,936,966')"

## 5. Smoke test — 1 volume do val

Antes de gastar 1h na pré-computação completa, valida que o pipeline ponta-a-ponta funciona no Kaggle. Esperado: ~30s, salva 1 arquivo em `/kaggle/working/recons/val/`.

In [ ]:
!python scripts/precompute_reconstructions.py --split val --max-volumes 1 --device cuda

In [ ]:
# Verifica o conteudo do .npz salvo
import numpy as np
from pathlib import Path

files = list(Path('/kaggle/working/recons/val').glob('*.npz'))
assert files, 'Smoke test nao gerou nenhum arquivo'
data = np.load(files[0])
print(f'Arquivo: {files[0].name}')
print(f'  recon:           {data["recon"].shape} {data["recon"].dtype}')
print(f'  target:          {data["target"].shape} {data["target"].dtype}')
print(f'  error_map:       {data["error_map"].shape} {data["error_map"].dtype}')
print(f'  max_val:         {float(data["max_val"]):.6f}')
print(f'  volume_id:       {str(data["volume_id"])}')
print(f'  split:           {str(data["split"])}')
print(f'  acceleration:    {int(data["acceleration"])}')
print(f'  center_fraction: {float(data["center_fraction"]):.4f}')
print(f'  varnet_sha256:   {str(data["varnet_sha256"])[:16]}...')
print(f'  size on disk:    {files[0].stat().st_size / 1e6:.1f} MB')

## 6. Pré-computação completa

Em ordem crescente de tamanho — se algo der errado, descobrimos cedo com o split menor.

| Split | Volumes | Tempo estimado (T4) |
|---|---|---|
| val  | 46  | ~7 min |
| cal  | 46  | ~7 min |
| test | 47  | ~7 min |
| train | 213 | ~32 min |
| **Total** | **352** | **~53 min** |

Os scripts são resumíveis — se a sessão for interrompida, basta rodar a célula de novo e ela pula os volumes já salvos.

In [ ]:
!python scripts/precompute_reconstructions.py --split val --device cuda --num-workers 2

In [ ]:
!python scripts/precompute_reconstructions.py --split cal --device cuda --num-workers 2

In [ ]:
!python scripts/precompute_reconstructions.py --split test --device cuda --num-workers 2

In [ ]:
!python scripts/precompute_reconstructions.py --split train --device cuda --num-workers 2

## 7. Verifica os outputs

In [ ]:
from pathlib import Path

RECONS = Path('/kaggle/working/recons')
EXPECTED = {'train': 213, 'val': 46, 'cal': 46, 'test': 47}

total_files = 0
total_size = 0.0
ok = True
for split, expected in EXPECTED.items():
    d = RECONS / split
    files = list(d.glob('*.npz'))
    n = len(files)
    size_gb = sum(f.stat().st_size for f in files) / 1e9
    total_files += n
    total_size += size_gb
    status = 'OK' if n == expected else f'FALTA {expected - n}'
    print(f'{split:6s}: {n:3d}/{expected:3d} volumes, {size_gb:6.2f} GB  [{status}]')
    if n != expected:
        ok = False

print(f'TOTAL:  {total_files}/352 volumes, {total_size:.2f} GB')
assert ok, 'Faltam volumes — re-rode as celulas do split que ficou incompleto'

In [ ]:
# Inspeciona o manifest agregado
import json
manifest = json.loads(Path('/kaggle/working/recons/precompute_manifest.json').read_text())
print(f'{len(manifest)} runs registrados')
for i, entry in enumerate(manifest):
    splits = entry.get('splits_requested', [])
    dt = entry.get('total_seconds', 0)
    proc = sum(r['processed'] for r in entry.get('reports', []))
    print(f'  Run {i+1}: splits={splits}, processados={proc}, {dt:.1f}s')

## 8. Próximo passo — salvar como Kaggle Dataset

Para que o S5 (treino do quantile network) reutilize estas reconstruções sem refazer a inferência:

1. No menu superior do notebook, clica em **File → Save Version**.
2. Escolhe **Save & Run All (Commit)**.
3. Quando terminar, vai em **Output → Create Dataset** (ou Save Output as Dataset).
4. Nomeia como `tcc-mri-recons-varnet-brain-4x`.
5. Configura como **privado**.

Esse dataset (~14 GB) vira o input do notebook do S5.